# Execution and source notes

Executed work from the IBM Applied Data Science Capstone labs. Unexecuted starter code cells were removed for a clear run order. Original instructions and authorship remain.

<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [2]:
%pip install beautifulsoup4 requests pandas

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

print("Web-scraping libraries loaded.")

Note: you may need to restart the kernel to use updated packages.
Web-scraping libraries loaded.


and we will provide some helper functions for you to process web scraped HTML table


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [3]:
static_url = (
    "https://en.wikipedia.org/w/index.php?"
    "title=List_of_Falcon_9_and_Falcon_Heavy_launches"
    "&oldid=1027686922"
)

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(
    static_url,
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
print("Page title:", soup.title.get_text() if soup.title else "No title")


Status code: 200
Page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia


Create a `BeautifulSoup` object from the HTML `response`


Print the page title to verify if the `BeautifulSoup` object was created properly 


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [4]:
html_tables = soup.find_all("table")

launch_tables = []

for table in html_tables:
    header_text = " ".join(
        cell.get_text(" ", strip=True)
        for cell in table.find_all("th", scope="col")
    )

    if "Flight No." in header_text and "Payload" in header_text:
        launch_tables.append(table)

print("Total HTML tables:", len(html_tables))
print("Launch tables found:", len(launch_tables))


Total HTML tables: 29
Launch tables found: 9


Starting from the third table is our target table contains the actual launch records.


You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [5]:
first_launch_table = launch_tables[0]
column_names = []

for header in first_launch_table.find_all("th", scope="col"):
    clean_header = BeautifulSoup(str(header), "html.parser")

    for reference in clean_header.find_all("sup"):
        reference.decompose()

    name = clean_header.get_text(" ", strip=True)
    name = re.sub(r"\s+", " ", name).strip()

    if name:
        column_names.append(name)

print(column_names)


['Flight No.', 'Date and time ( UTC )', 'Version, Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


Check the extracted column names


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [6]:
def clean_cell(cell):
    cleaned = BeautifulSoup(str(cell), "html.parser")

    for reference in cleaned.find_all("sup"):
        reference.decompose()

    text = cleaned.get_text(" ", strip=True)
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", text).strip()


records = []

for table in launch_tables:
    for row in table.find_all("tr"):
        flight_header = row.find("th", scope="row")

        if flight_header is None:
            continue

        flight_number = clean_cell(flight_header)

        if not flight_number.isdigit():
            continue

        cells = row.find_all("td", recursive=False)

        if len(cells) != 9:
            continue

        values = [clean_cell(cell) for cell in cells]

        records.append({
            "Flight No.": int(flight_number),
            "Date and time (UTC)": values[0],
            "Version Booster": values[1],
            "Launch site": values[2],
            "Payload": values[3],
            "Payload mass": values[4],
            "Orbit": values[5],
            "Customer": values[6],
            "Launch outcome": values[7],
            "Booster landing": values[8]
        })

df = pd.DataFrame(records)

print("Launch records extracted:", len(df))
display(df.head())

Launch records extracted: 121


,Flight No.,Date and time (UTC),Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,"4 June 2010, 18:45",F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,,LEO,SpaceX,Success,Failure (parachute)
1,2,"8 December 2010, 15:43",F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute)
2,3,"22 May 2012, 07:44",F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),"525 kg (1,157 lb)",LEO ( ISS ),NASA ( COTS ),Success,No attempt
3,4,"8 October 2012, 00:35",F9 v1.0 B0006.1,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg (10,400 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt
4,5,"1 March 2013, 15:10",F9 v1.0 B0007.1,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg (10,752 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [7]:
print("Table size:", df.shape)
print("Duplicate flight numbers:", df["Flight No."].duplicated().sum())

df.to_csv("spacex_web_scraped.csv", index=False)

print("Saved spacex_web_scraped.csv")
display(df.tail())

Table size: (121, 10)
Duplicate flight numbers: 0
Saved spacex_web_scraped.csv


,Flight No.,Date and time (UTC),Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
116,117,9 May 2021 06:42,F9 B5 ♺ B1051.10,"CCSFS , SLC-40",Starlink 27 v1.0 (60 satellites),"15,600 kg (34,400 lb)",LEO,SpaceX,Success,Success (drone ship)
117,118,15 May 2021 22:56,F9 B5 ♺ B1058.8,"KSC , LC-39A",Starlink 26 v1.0 (52 Satellites) Capella -6 & ...,"~ 14,000 kg (31,000 lb)",LEO,SpaceX Capella Space and Tyvak,Success,Success (drone ship)
118,119,26 May 2021 18:59,F9 B5 ♺ B1063.2,"CCSFS , SLC-40",Starlink 28 v1.0 (60 Satellites),"15,600 kg (34,400 lb)",LEO,SpaceX,Success,Success (drone ship)
119,120,3 June 2021 17:29,F9 B5 B1067.1,"KSC , LC-39A",SpaceX CRS-22 ( Dragon C209.1 ),"3,328 kg (7,337 lb)",LEO ( ISS ),NASA ( CRS ),Success,Success (drone ship)
120,121,6 June 2021 04:26,F9 B5 ♺ B1061.3,"CCSFS , SLC-40",SXM-8,"7,000 kg (15,000 lb)",GTO,Sirius XM,Success,Success (drone ship)


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
